# PySpark reference 6: complex types and JSON

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/pyspark/pyspark-reference-06-complex-types-and-json.ipynb)

Arrays, maps and structs, higher-order functions, exploding and `inline`, building and flattening nested records, JSON parsing and generation, and the VARIANT type. Practise it in [PySpark practice 6: complex types and JSON](pyspark-practice-06-complex-types-and-json.ipynb).

*Last reviewed: November 2026, against Apache Spark 4.0.*

In [ ]:
# Setup: installs the packages this notebook needs when they are missing (e.g. in Colab).
!pip install -q pyspark~=4.0.0

## Setup and sample data

A local session, the shared `customers`, `orders`, `order_items`, `products` and `events` tables (customer 4 has no region, customers 7 and 8 never ordered, `SKU-4014` has no product row, one event is an exact retry), and small helpers: `plan_lines(df)` returns what `explain()` prints, `final_plan(df)` runs a query and returns the plan Spark actually used, and `expected_error()` prints the error class of an example that fails on purpose. It also creates `contacts`, `baskets` (arrays, with one empty basket) and `payloads` (JSON, one invalid).

In [ ]:
import os
import sys
import tempfile
import time

from pyspark.sql import SparkSession

os.environ["PYSPARK_PYTHON"] = sys.executable  # Python workers use this notebook's interpreter
os.environ["TZ"] = "UTC"  # the sample timestamps are UTC
time.tzset()
WORK_DIR = tempfile.mkdtemp(prefix="pyspark-")  # scratch space for the files the lessons write

spark = (
    SparkSession.builder.master("local[2]")
    .config("spark.sql.shuffle.partitions", "4")  # the default of 200 is far too many for tiny data
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.warehouse.dir", os.path.join(WORK_DIR, "warehouse"))
    .config("spark.ui.enabled", "false")
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version)

In [ ]:
from datetime import date, datetime
from decimal import Decimal as D

from pyspark.sql import Window
from pyspark.sql import functions as F

customers = spark.createDataFrame([
    (1, "Acme Corp", "North", date(2024, 1, 3)),
    (2, "Bright Retail", "North", date(2024, 2, 11)),
    (3, "Cyan Traders", "South", date(2024, 3, 15)),
    (4, "Delta Grocers", None, date(2024, 5, 2)),  # region never arrived from the CRM
    (5, "Echo Supplies", "West", date(2024, 6, 20)),
    (6, "Foxtrot Foods", "West", date(2024, 8, 1)),
    (7, "Gamma Sports", "East", date(2024, 9, 14)),  # 7 and 8 have never ordered
    (8, "Hotel Homeware", "East", date(2024, 11, 5)),
], "customer_id int, name string, region string, signup_date date")

orders = spark.createDataFrame([
    (1, 1, "paid", datetime(2026, 1, 5, 9, 15), D("120.00")),
    (2, 1, "shipped", datetime(2026, 1, 5, 10, 5), D("84.50")),
    (3, 1, "returned", datetime(2026, 1, 5, 11, 40), D("30.00")),
    (4, 2, "paid", datetime(2026, 1, 6, 15, 20), D("260.00")),
    (5, 2, "placed", datetime(2026, 1, 7, 8, 5), D("55.25")),
    (6, 3, "shipped", datetime(2026, 1, 7, 17, 45), D("149.00")),
    (7, 3, "paid", datetime(2026, 1, 9, 12, 10), D("310.00")),
    (8, 4, "returned", datetime(2026, 1, 10, 19, 30), D("89.90")),
    (9, 4, "shipped", datetime(2026, 1, 12, 8, 55), D("1180.00")),
    (10, 5, "paid", datetime(2026, 1, 12, 13, 0), D("74.00")),
    (11, 6, "placed", datetime(2026, 1, 13, 10, 25), D("210.00")),
    (12, 6, "shipped", datetime(2026, 1, 14, 0, 30), D("96.40")),  # just after midnight UTC
], "order_id int, customer_id int, status string, order_ts timestamp, amount decimal(12,2)")

order_items = spark.createDataFrame([
    (1, "SKU-4001", 2, D("12.50")), (1, "SKU-4007", 1, D("95.00")),
    (2, "SKU-4002", 1, D("34.50")), (2, "SKU-4009", 2, D("25.00")),
    (3, "SKU-4002", 1, D("30.00")),
    (4, "SKU-4003", 1, D("149.00")), (4, "SKU-4004", 2, D("55.50")),
    (5, "SKU-4005", 1, D("55.25")),
    (6, "SKU-4003", 1, D("149.00")),
    (7, "SKU-4006", 1, D("189.00")), (7, "SKU-4008", 1, D("121.00")),
    (8, "SKU-4010", 1, D("89.90")),
    (9, "SKU-4011", 20, D("59.00")),
    (10, "SKU-4012", 2, D("37.00")),
    (11, "SKU-4013", 1, D("210.00")),
    (12, "SKU-4014", 1, D("41.40")), (12, "SKU-4005", 1, D("55.00")),
], "order_id int, sku string, qty int, unit_price decimal(8,2)")

products = spark.createDataFrame([
    ("SKU-4001", "Kitchen", D("12.50")), ("SKU-4002", "Kitchen", D("34.50")),
    ("SKU-4003", "Garden", D("149.00")), ("SKU-4004", "Garden", D("55.50")),
    ("SKU-4005", "Pantry", D("55.25")), ("SKU-4006", "Outdoor", D("189.00")),
    ("SKU-4007", "Kitchen", D("95.00")), ("SKU-4008", "Outdoor", D("121.00")),
    ("SKU-4009", "Pantry", D("25.00")), ("SKU-4010", "Garden", D("89.90")),
    ("SKU-4011", "Pantry", D("59.00")), ("SKU-4012", "Kitchen", D("37.00")),
    ("SKU-4013", "Outdoor", D("210.00")),  # SKU-4014 is missing
], "sku string, category string, list_price decimal(8,2)")

events = spark.createDataFrame([
    (1, "view", datetime(2026, 1, 6, 9, 0)), (1, "add_to_cart", datetime(2026, 1, 6, 9, 2)),
    (1, "purchase", datetime(2026, 1, 6, 9, 10)), (2, "view", datetime(2026, 1, 7, 10, 0)),
    (2, "purchase", datetime(2026, 1, 7, 10, 40)), (3, "view", datetime(2026, 1, 6, 11, 15)),
    (3, "view", datetime(2026, 1, 8, 11, 20)), (4, "add_to_cart", datetime(2026, 1, 9, 20, 5)),
    (4, "purchase", datetime(2026, 1, 10, 8, 30)), (5, "view", datetime(2026, 1, 11, 14, 45)),
    (5, "view", datetime(2026, 1, 12, 9, 0)), (5, "purchase", datetime(2026, 1, 12, 12, 30)),
    (5, "view", datetime(2026, 1, 12, 9, 0)),  # an SDK retry committed this event twice
], "user_id int, event string, ts timestamp")

import contextlib
import io
import logging
import re


def plan_lines(df, mode="simple"):
    """The lines df.explain(mode) prints."""
    buffer = io.StringIO()
    with contextlib.redirect_stdout(buffer):
        df.explain(mode)
    return buffer.getvalue().splitlines()


def final_plan(df):
    """Run df, then return the plan Spark actually used (AQE's final plan when there is one)."""
    df.collect()
    text = "\n".join(plan_lines(df))
    if "== Final Plan ==" in text:
        text = text.split("== Final Plan ==")[1].split("== Initial Plan ==")[0]
    return text


@contextlib.contextmanager
def expected_error():
    """Run a block that is meant to fail and print the error class instead of a stack trace."""
    spark.sparkContext.setLogLevel("OFF")
    for logger in ("DataFrameQueryContextLogger", "SQLQueryContextLogger"):
        logging.getLogger(logger).setLevel(logging.CRITICAL)
    try:
        yield
    except Exception as error:
        print(type(error).__name__ + ":", str(error).split(".")[0][:100])
    finally:
        spark.sparkContext.setLogLevel("ERROR")

contacts = spark.createDataFrame([(1, "555-01", None), (2, None, "555-02"), (3, None, None)], "id int, mobile string, home string")
baskets = spark.createDataFrame(
    [(1, ["milk", "apple", "milk"], [1, 3, 2]), (2, ["bread"], [5]), (3, [], [])],
    "order_id int, items array<string>, qty array<int>",
)
payloads = spark.createDataFrame(
    [(1, '{"device": "ios", "geo": {"country": "DE"}, "tags": ["new", "promo"]}'),
     (2, '{"device": "web", "geo": {"country": "US"}}'),
     (3, "not json")],
    "event_id int, body string",
)

## Contents

| section | entries |
| --- | --- |
| [Array functions](#array-functions) | [`array`](#array), [`size`, `array_contains`](#size-array_contains), [`array_distinct`, `sort_array`](#array_distinct-sort_array), [`array_join`](#array_join), [`flatten`](#flatten), [`arrays_zip`](#arrays_zip), [`element_at`, `try_element_at`, `get`](#element_at-try_element_at-get), [`transform`](#transform), [`filter`](#filter), [`exists`, `forall`](#exists-forall), [`aggregate`](#aggregate) |
| [Struct and map functions](#struct-and-map-functions) | [`struct`](#struct), [`create_map`](#create_map), [`map_keys`, `map_values`, `map_contains_key`](#map_keys-map_values-map_contains_key) |
| [Generator functions](#generator-functions) | [`explode`, `explode_outer`](#explode-explode_outer), [`posexplode`](#posexplode) |
| [More array and map functions](#more-array-and-map-functions) | [`array_union`, `array_intersect`, `array_except`, `arrays_overlap`](#array_union-array_intersect-array_except-arrays_overlap), [`slice`, `array_position`, `array_remove`, `array_compact`](#slice-array_position-array_remove-array_compact), [`array_max`, `array_min`, `array_sort`, `zip_with`](#array_max-array_min-array_sort-zip_with), [`map_from_arrays`, `map_from_entries`, `map_entries`, `map_concat`](#map_from_arrays-map_from_entries-map_entries-map_concat), [`transform_keys`, `transform_values`, `map_filter`](#transform_keys-transform_values-map_filter), [`inline`, `inline_outer`](#inline-inline_outer) |
| [Nested records](#nested-records) | [Building nested output with `collect_list` of structs](#building-nested-output-with-collect_list-of-structs), [Flattening a nested schema](#flattening-a-nested-schema) |
| [JSON functions](#json-functions) | [`schema_of_json`](#schema_of_json), [`from_json`](#from_json), [`to_json`](#to_json), [`get_json_object`](#get_json_object), [`json_tuple`](#json_tuple) |
| [VARIANT](#variant) | [VARIANT: `parse_json`, `variant_get`, `try_variant_get`, `schema_of_variant`](#variant-parse_json-variant_get-try_variant_get-schema_of_variant) |

## Array functions

Build, inspect and transform array columns. The higher-order functions (`transform`, `filter`, `exists`,
`forall`, `aggregate`) take a Python lambda that Spark turns into a SQL expression, so no Python runs per row.
The examples use `baskets`: parallel `items` and `qty` arrays, with one empty basket.

| function | what it does |
| --- | --- |
| `array` | Build an array from several columns or literals |
| `size`, `array_contains` | Number of elements, and whether the array holds a value |
| `array_distinct`, `sort_array` | Remove duplicate elements; sort the elements (ascending by default) |
| `array_join` | Join array elements into one string with a separator |
| `flatten` | Turn an array of arrays into one array |
| `arrays_zip` | Pair several arrays element by element into an array of structs |
| `element_at`, `try_element_at`, `get` | Read one element: `element_at` and `try_element_at` are 1-based (negative counts from the end); `get` is 0-based |
| `transform` | Apply a function to every element |
| `filter` | Keep the elements for which a condition is true |
| `exists`, `forall` | Whether any element, or every element, satisfies a condition |
| `aggregate` | Reduce an array to one value with a start value and a merge function |

### `array`

Build an array from several columns or literals.

**Signature:** `F.array(*cols)`

**Use it for:** Collecting related fields, such as several phone numbers, into one column.

**Notes:** All elements must have a compatible type.

In [ ]:
contacts.select("id", F.array("mobile", "home").alias("phones")).show()

### `size`, `array_contains`

Number of elements, and whether the array holds a value.

**Signature:** `F.size(col)` · `F.array_contains(col, value)`

**Use it for:** Filtering on list contents and length checks.

**Notes:** `size` counts null elements too; the size of a null array is null in ANSI mode (-1 when ANSI is off).

In [ ]:
baskets.select("order_id", "items", F.size("items").alias("size"), F.array_contains("items", "milk").alias("has_milk")).show()

### `array_distinct`, `sort_array`

Remove duplicate elements; sort the elements (ascending by default).

**Signature:** `F.array_distinct(col)` · `F.sort_array(col, asc=True)`

**Use it for:** Tidy, comparable lists of tags.

**Notes:** `sort_array` keeps duplicates; combine the two for a sorted distinct list.

In [ ]:
baskets.select("items", F.array_distinct("items").alias("distinct"), F.sort_array("items").alias("sorted"),
               F.sort_array("items", asc=False).alias("sorted_desc")).show(truncate=False)

### `array_join`

Join array elements into one string with a separator.

**Signature:** `F.array_join(col, sep, null_replacement=None)`

**Use it for:** Readable exports of list columns.

**Notes:** Null elements are skipped unless you pass `null_replacement`.

In [ ]:
baskets.select("order_id", F.array_join("items", ", ").alias("joined")).show()

### `flatten`

Turn an array of arrays into one array.

**Signature:** `F.flatten(col)`

**Use it for:** Combining nested lists, for example after `collect_list` of arrays.

**Notes:** Only one level is removed per call.

In [ ]:
spark.range(1).select(F.flatten(F.array(F.array(F.lit(1), F.lit(2)), F.array(F.lit(3)))).alias("flatten")).show()

### `arrays_zip`

Pair several arrays element by element into an array of structs.

**Signature:** `F.arrays_zip(*cols)`

**Use it for:** Exploding parallel arrays (items and quantities) together, without mixing them up.

**Notes:** The shorter array is padded with nulls.

In [ ]:
baskets.select("order_id", F.arrays_zip("items", "qty").alias("lines")).show(truncate=False)

### `element_at`, `try_element_at`, `get`

Read one element: `element_at` and `try_element_at` are 1-based (negative counts from the end); `get` is 0-based.

**Signature:** `F.element_at(col, index)` · `F.try_element_at(col, F.lit(index))` · `F.get(col, index)`

| parameter | type | description |
| --- | --- | --- |
| `col` | Column or name | array or map |
| `index / key` | int or Column | 1-based index (negative from the end) for arrays; key for maps |

**Use it for:** The first or last element of a list.

**Notes:** In ANSI mode `element_at` raises an error past the end of an array; `try_element_at` and `get` return null.

In [ ]:
baskets.select("items", F.try_element_at("items", F.lit(1)).alias("first"), F.try_element_at("items", F.lit(-1)).alias("last"),
               F.get("items", 0).alias("get_0")).show()

### `transform`

Apply a function to every element.

**Signature:** `F.transform(col, lambda x: ...)`

| parameter | type | description |
| --- | --- | --- |
| `col` | Column or name | array |
| `f` | function | `lambda x: ...` or `lambda x, i: ...` (element and index), returning a Column |

**Use it for:** Cleaning every tag or converting every price.

**Notes:** Use Spark functions inside the lambda (`F.trim(x)`), not Python string methods.

In [ ]:
baskets.select("qty", F.transform("qty", lambda q: q * 10).alias("qty_x10"), F.transform("items", lambda i: F.upper(i)).alias("upper_items")).show(truncate=False)

### `filter`

Keep the elements for which a condition is true.

**Signature:** `F.filter(col, lambda x: condition)`

| parameter | type | description |
| --- | --- | --- |
| `col` | Column or name | array |
| `f` | function | `lambda x: condition` (or `lambda x, i: ...`) |

**Use it for:** Dropping empty or invalid elements from a list.

**Notes:** It shares its name with `DataFrame.filter`; here it works inside one array.

In [ ]:
baskets.select("qty", F.filter("qty", lambda q: q >= 2).alias("qty_ge_2")).show()

### `exists`, `forall`

Whether any element, or every element, satisfies a condition.

**Signature:** `F.exists(col, lambda x: ...)` · `F.forall(col, lambda x: ...)`

**Use it for:** "does this order contain a restricted item?" style checks.

**Notes:** For an empty array `exists` is false and `forall` is true.

In [ ]:
baskets.select("items", F.exists("items", lambda i: i == "bread").alias("has_bread"), F.forall("qty", lambda q: q > 0).alias("all_positive")).show()

### `aggregate`

Reduce an array to one value with a start value and a merge function.

**Signature:** `F.aggregate(col, initial, lambda acc, x: ...)`

| parameter | type | description |
| --- | --- | --- |
| `col` | Column or name | array |
| `initialValue` | Column | start value; sets the accumulator type |
| `merge` | function | `lambda acc, x: ...` |
| `finish` | function, optional | `lambda acc: ...` applied at the end |

**Use it for:** Totals, products or custom reductions over a list.

**Notes:** The start value sets the result type: start a decimal sum with a decimal zero.

In [ ]:
baskets.select("qty", F.aggregate("qty", F.lit(0), lambda total, q: total + q).alias("total_qty")).show()

## Struct and map functions

A **struct** groups named fields into one column; a **map** holds key-value pairs whose keys can differ per row.

| function | what it does |
| --- | --- |
| `struct` | Group several columns into one struct column |
| `create_map` | Build a map from alternating key and value columns |
| `map_keys`, `map_values`, `map_contains_key` | The keys or values of a map as arrays, and whether a key exists |

### `struct`

Group several columns into one struct column. Read a field back with a dot.

**Signature:** `F.struct(*cols)`

| parameter | type | description |
| --- | --- | --- |
| `*cols` | Column or name | fields; names come from column names or aliases |

**Use it for:** Nested output for JSON, and carrying several fields through `max_by` or `collect_list` together.

**Notes:** Field names come from the column names or aliases.

In [ ]:
profiles = customers.select("customer_id", F.struct("name", "region").alias("profile"))
profiles.select("customer_id", "profile", F.col("profile.name").alias("name")).show(3, truncate=False)

### `create_map`

Build a map from alternating key and value columns.

**Signature:** `F.create_map(key1, value1, key2, value2, ...)`

| parameter | type | description |
| --- | --- | --- |
| `*cols` | Column | alternating keys and values: `key1, value1, key2, value2, ...` |

**Use it for:** Small lookup vocabularies and attribute bags.

**Notes:** Keys must not be null, and all values share one type.

In [ ]:
status_names = F.create_map(F.lit("paid"), F.lit("Paid"), F.lit("shipped"), F.lit("On its way"))
orders.select("status", F.element_at(status_names, F.col("status")).alias("label")).distinct().orderBy("status").show()

### `map_keys`, `map_values`, `map_contains_key`

The keys or values of a map as arrays, and whether a key exists.

**Signature:** `F.map_keys(col)` · `F.map_values(col)` · `F.map_contains_key(col, key)`

**Use it for:** Inspecting attribute bags and telling a missing key from a null value.

**Notes:** `element_at` returns null both for a missing key and for a key whose value is null.

In [ ]:
attrs = customers.select("customer_id", F.create_map(F.lit("region"), F.col("region"), F.lit("signup"), F.col("signup_date").cast("string")).alias("attrs"))
attrs.where("customer_id IN (1, 4)").select("customer_id", F.map_keys("attrs").alias("keys"), F.map_values("attrs").alias("values"),
    F.map_contains_key("attrs", F.lit("region")).alias("has_region_key")).show(truncate=False)

## Generator functions

Turn one row into many: one output row per array element or map entry.

| function | what it does |
| --- | --- |
| `explode`, `explode_outer` | One row per element |
| `posexplode` | Like `explode`, plus the element's position (0-based) |

### `explode`, `explode_outer`

One row per element. `explode` drops rows whose array is empty or null; `explode_outer` keeps them with a null element.

**Signature:** `F.explode(col)` · `F.explode_outer(col)`

| parameter | type | description |
| --- | --- | --- |
| `col` | Column or name | array (one row per element) or map (one row per entry, columns `key` and `value`) |

**Use it for:** Flattening order lines, tags and event lists into rows.

**Notes:** The row count multiplies; and `explode` silently loses parents with empty lists.

In [ ]:
baskets.select("order_id", F.explode("items").alias("item")).show()
baskets.select("order_id", F.explode_outer("items").alias("item")).where("order_id = 3").show()

### `posexplode`

Like `explode`, plus the element's position (0-based).

**Signature:** `F.posexplode(col)` (returns `pos` and `col`)

**Use it for:** Keeping the original order of list elements, such as line numbers.

**Notes:** Rename the two outputs with `.alias("pos", "item")`.

In [ ]:
baskets.select("order_id", F.posexplode("items").alias("pos", "item")).show()

## More array and map functions

Set operations, slicing and searching, element-wise functions, map construction and higher-order map functions, and exploding arrays of structs.

| function | what it does |
| --- | --- |
| `array_union`, `array_intersect`, `array_except`, `arrays_overlap` | Set operations between two arrays, returning distinct elements; `arrays_overlap` tests whether they share any non-null element |
| `slice`, `array_position`, `array_remove`, `array_compact` | Take a sub-array (`slice(arr, start, length)`, 1-based), find the 1-based position of a value (0 if absent), remove all occurrences of a value, and drop nulls |
| `array_max`, `array_min`, `array_sort`, `zip_with` | Largest and smallest element, a sorted copy (nulls last), and an element-wise combination of two arrays with a function |
| `map_from_arrays`, `map_from_entries`, `map_entries`, `map_concat` | Build a map from a keys array and a values array, or from an array of `(key, value)` structs; turn a map back into entries; merge maps |
| `transform_keys`, `transform_values`, `map_filter` | Higher-order functions for maps: rewrite each key or value with `lambda k, v: ...`, or keep entries matching a condition |
| `inline`, `inline_outer` | Explode an array of structs into rows **and** columns in one step: one row per element, one column per struct field |

### `array_union`, `array_intersect`, `array_except`, `arrays_overlap`

Set operations between two arrays, returning distinct elements; `arrays_overlap` tests whether they share any non-null element.

**Signature:** `F.array_union(a, b)` · `F.array_intersect(a, b)` · `F.array_except(a, b)` · `F.arrays_overlap(a, b)`

**Use it for:** Comparing tag lists, permissions or product sets between two columns.

**Notes:** Results are de-duplicated; element order follows the first array.

In [ ]:
sets = spark.createDataFrame([(["a", "b", "c"], ["b", "d"])], "x array<string>, y array<string>")
sets.select(F.array_union("x", "y").alias("union"), F.array_intersect("x", "y").alias("intersect"),
            F.array_except("x", "y").alias("x_minus_y"), F.arrays_overlap("x", "y").alias("overlap")).show()

### `slice`, `array_position`, `array_remove`, `array_compact`

Take a sub-array (`slice(arr, start, length)`, 1-based), find the 1-based position of a value (0 if absent), remove all occurrences of a value, and drop nulls.

**Signature:** `F.slice(col, start, length)` · `F.array_position(col, value)` · `F.array_remove(col, value)` · `F.array_compact(col)`

**Use it for:** Top-N lists, cleaning lists before joining them to strings.

**Notes:** `array_position` returns 0, not null, when the value is missing.

In [ ]:
lists = spark.createDataFrame([(["a", None, "b", "a", "c"],)], "x array<string>")
lists.select(F.slice("x", 1, 3).alias("first_3"), F.array_position("x", "b").alias("pos_b"), F.array_position("x", "z").alias("pos_z"),
             F.array_remove("x", "a").alias("without_a"), F.array_compact("x").alias("no_nulls")).show(truncate=False)

### `array_max`, `array_min`, `array_sort`, `zip_with`

Largest and smallest element, a sorted copy (nulls last), and an element-wise combination of two arrays with a function.

**Signature:** `F.array_max(col)` · `F.array_sort(col)` · `F.zip_with(a, b, lambda x, y: ...)`

**Use it for:** Per-row statistics over a list, and computing line totals from parallel price and quantity arrays.

**Notes:** `zip_with` pads the shorter array with nulls.

In [ ]:
prices = spark.createDataFrame([([2.5, 10.0, 4.0], [2, 1, 3])], "price array<double>, qty array<int>")
prices.select(F.array_max("price").alias("max"), F.array_sort("price").alias("sorted"),
              F.zip_with("price", "qty", lambda p, q: p * q).alias("line_totals")).show(truncate=False)

### `map_from_arrays`, `map_from_entries`, `map_entries`, `map_concat`

Build a map from a keys array and a values array, or from an array of `(key, value)` structs; turn a map back into entries; merge maps.

**Signature:** `F.map_from_arrays(keys, values)` · `F.map_from_entries(col)` · `F.map_entries(col)` · `F.map_concat(*cols)`

**Use it for:** Pivot-like attribute bags and merging attribute maps from two sources.

**Notes:** Duplicate keys raise an error by default (`spark.sql.mapKeyDedupPolicy=EXCEPTION`); set it to `LAST_WIN` to keep the last.

In [ ]:
kv = spark.createDataFrame([(["color", "size"], ["red", "M"])], "k array<string>, v array<string>")
built = kv.select(F.map_from_arrays("k", "v").alias("attrs"))
built.select("attrs", F.map_entries("attrs").alias("entries"),
             F.map_concat("attrs", F.create_map(F.lit("fit"), F.lit("slim"))).alias("merged")).show(truncate=False)

### `transform_keys`, `transform_values`, `map_filter`

Higher-order functions for maps: rewrite each key or value with `lambda k, v: ...`, or keep entries matching a condition.

**Signature:** `F.transform_values(col, lambda k, v: ...)` · `F.map_filter(col, lambda k, v: ...)`

**Use it for:** Normalizing attribute keys (lower case) and dropping empty attributes.

**Notes:** Rewriting keys into duplicates raises an error under the default dedup policy.

In [ ]:
bag = spark.createDataFrame([({"Color": "red", "Size": "", "Fit": "slim"},)], "attrs map<string,string>")
bag.select(F.transform_keys("attrs", lambda k, v: F.lower(k)).alias("lower_keys"),
           F.map_filter("attrs", lambda k, v: v != "").alias("non_empty"),
           F.transform_values("attrs", lambda k, v: F.upper(v)).alias("upper_values")).show(truncate=False)

### `inline`, `inline_outer`

Explode an array of structs into rows **and** columns in one step: one row per element, one column per struct field. `inline_outer` keeps rows whose array is empty or null.

**Signature:** `F.inline(col)` · `F.inline_outer(col)`

**Use it for:** Flattening order lines, addresses or any list of records stored as an array of structs.

**Notes:** The output columns take the struct field names; rename them afterwards if they clash.

In [ ]:
nested = spark.createDataFrame([(1, [("SKU-1", 2), ("SKU-2", 1)]), (2, [])], "order_id int, lines array<struct<sku:string,qty:int>>")
nested.select("order_id", F.inline_outer("lines")).show()

## Nested records

Building nested output, flattening nested input.

| function | what it does |
| --- | --- |
| Building nested output with `collect_list` of structs | The reverse of exploding: group rows and collect structs into an array to produce one nested record per key, for example one JSON document per order |
| Flattening a nested schema | Turn every struct field into a top-level column named `parent_child`, recursively, by walking `df.schema` |

### Building nested output with `collect_list` of structs

The reverse of exploding: group rows and collect structs into an array to produce one nested record per key, for example one JSON document per order.

**Signature:** `groupBy(key).agg(F.collect_list(F.struct(...)))`

**Use it for:** APIs and document stores that expect nested records.

**Notes:** `collect_list` order is not guaranteed; sort the array with `array_sort` on a struct whose first field is the sort key.

In [ ]:
docs = (order_items.where("order_id <= 2")
        .groupBy("order_id")
        .agg(F.array_sort(F.collect_list(F.struct("sku", "qty", "unit_price"))).alias("lines")))
docs.select(F.to_json(F.struct("order_id", "lines")).alias("document")).orderBy("document").show(truncate=False)

### Flattening a nested schema

Turn every struct field into a top-level column named `parent_child`, recursively, by walking `df.schema`. Arrays are left as they are (explode them first if needed).

**Signature:** a small helper over `df.schema.fields` (shown below)

**Use it for:** Loading nested sources into warehouses and BI tools that expect flat tables.

**Notes:** Field names can collide after flattening (`a.b_c` and `a_b.c`); check for duplicates.

In [ ]:
from pyspark.sql.types import StructType

def flatten(df, sep="_"):
    def columns(schema, prefix=""):
        out = []
        for field in schema.fields:
            path = f"{prefix}{field.name}"
            if isinstance(field.dataType, StructType):
                out += columns(field.dataType, path + ".")
            else:
                out.append(F.col(path).alias(path.replace(".", sep)))
        return out
    return df.select(*columns(df.schema))

deep = spark.createDataFrame([(1, ((7, "Gamma"), "web"), ["new"])], "order_id int, meta struct<customer:struct<id:int,name:string>,channel:string>, tags array<string>")
flatten(deep).show()

## JSON functions

Parse JSON text into typed columns and write columns back as JSON. The examples use `payloads`, with one
complete event, one without tags and one that is not JSON.

| function | what it does |
| --- | --- |
| `schema_of_json` | Infer a schema string from one JSON example |
| `from_json` | Parse JSON text into a struct (or array or map) using a schema |
| `to_json` | Write a struct, array or map column as JSON text |
| `get_json_object` | Extract one value from JSON text by a JSONPath such as `$.geo.country`, as a string |
| `json_tuple` | Extract several top-level keys at once, as strings |

### `schema_of_json`

Infer a schema string from one JSON example.

**Signature:** `F.schema_of_json(F.lit(sample))`

**Use it for:** A quick start for writing the schema `from_json` needs.

**Notes:** One sample can miss optional fields; review the result before using it.

In [ ]:
sample = payloads.first().body
print(spark.range(1).select(F.schema_of_json(F.lit(sample))).first()[0])

### `from_json`

Parse JSON text into a struct (or array or map) using a schema.

**Signature:** `F.from_json(col, schema)`

| parameter | type | description |
| --- | --- | --- |
| `col` | Column or name | JSON string |
| `schema` | DDL string, StructType or Column | expected structure |
| `options` | dict, optional | e.g. `{"mode": "FAILFAST"}`, `{"timestampFormat": ...}`, `{"allowSingleQuotes": "true"}` |

**Use it for:** Typing event payloads once, then querying fields with dots.

**Notes:** Malformed text and mismatched schemas give nulls, not errors; count them after parsing.

In [ ]:
parsed = payloads.select("event_id", F.from_json("body", "device string, geo struct<country string>, tags array<string>").alias("data"))
parsed.select("event_id", "data.device", "data.geo.country", "data.tags").show()

### `to_json`

Write a struct, array or map column as JSON text.

**Signature:** `F.to_json(col)`

| parameter | type | description |
| --- | --- | --- |
| `col` | Column or name | struct, array or map |
| `options` | dict, optional | e.g. `{"ignoreNullFields": "false"}` |

**Use it for:** Messages for Kafka or an API; one JSON document per row.

**Notes:** Build the shape with `struct` first to control field names.

In [ ]:
orders.select(F.to_json(F.struct("order_id", "status", "amount")).alias("message")).show(3, truncate=False)

### `get_json_object`

Extract one value from JSON text by a JSONPath such as `$.geo.country`, as a string.

**Signature:** `F.get_json_object(col, "$.path")`

| parameter | type | description |
| --- | --- | --- |
| `col` | Column or name | JSON string |
| `path` | str | JSONPath: `$.a.b`, `$.items[0].sku`, `$.tags[*]` |

**Use it for:** Pulling one field without defining a schema.

**Notes:** Every call re-parses the text; for several fields, parse once with `from_json`.

In [ ]:
payloads.select("event_id", F.get_json_object("body", "$.geo.country").alias("country"), F.get_json_object("body", "$.tags[0]").alias("first_tag")).show()

### `json_tuple`

Extract several top-level keys at once, as strings.

**Signature:** `F.json_tuple(col, *keys)`

**Use it for:** Flat JSON where a schema would be overkill.

**Notes:** Only top-level keys, and only strings: nested objects come back as JSON text.

In [ ]:
payloads.select("event_id", F.json_tuple("body", "device", "geo").alias("device", "geo_text")).show(truncate=False)

## VARIANT

Semi-structured data without a fixed schema (Spark 4).

| function | what it does |
| --- | --- |
| VARIANT: `parse_json`, `variant_get`, `try_variant_get`, `schema_of_variant` | Spark 4's `variant` type stores semi-structured JSON-like data in a binary form that is faster to query than re-parsing strings, without fixing a schema up front |

### VARIANT: `parse_json`, `variant_get`, `try_variant_get`, `schema_of_variant`

Spark 4's `variant` type stores semi-structured JSON-like data in a binary form that is faster to query than re-parsing strings, without fixing a schema up front. Parse once with `parse_json`, then extract typed values with `variant_get(v, path, type)`.

**Signature:** `F.parse_json(col)` · `F.variant_get(v, "$.path", "int")` · `F.try_variant_get(...)` · `F.schema_of_variant(v)`

**Use it for:** Event payloads whose fields vary by event type, landed once and queried many ways.

**Notes:** `variant_get` raises on a value that cannot be cast to the requested type; `try_variant_get` returns null. Invalid JSON fails `parse_json` (use `try_parse_json`).

In [ ]:
raw_events = spark.createDataFrame([(1, '{"type": "click", "x": 10, "meta": {"page": "/home"}}'), (2, '{"type": "buy", "amount": "12.50"}')], "id int, body string")
v = raw_events.select("id", F.parse_json("body").alias("v"))
v.select("id", F.variant_get("v", "$.type", "string").alias("type"), F.try_variant_get("v", "$.x", "int").alias("x"),
         F.try_variant_get("v", "$.amount", "decimal(10,2)").alias("amount"), F.variant_get("v", "$.meta.page", "string").alias("page"),
         F.schema_of_variant("v").alias("schema")).show(truncate=False)

**Next:** practise these entries in [PySpark practice 6: complex types and JSON](pyspark-practice-06-complex-types-and-json.ipynb).